# Chapter 13 — Tracking-Based Promotion Decision

This notebook demonstrates how to use MLflow Tracking data to make
automated promotion decisions. We query the experiment for the
best-performing run, compare it against thresholds, and promote the
exact version that run evaluated — not simply the latest version.

**Continues from:** `03_evaluate_agent` (evaluation runs logged with metrics)


In [ ]:
%pip install mlflow
%restart_python


## 1. Set Up Variables


In [ ]:
UC_CATALOG = "dev_catalog"
UC_SCHEMA  = "finance"
try:
    username = spark.conf.get("spark.databricks.notebook.userName")
except:
    username = "unknown"
if username == "unknown":
    username = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Current user: {username}")


## 2. Query Experiment Runs and Promote the Winner

This turns promotion from a manual judgment call into an automated gate
in the CI/CD pipeline. We filter for runs that meet quality thresholds,
rank by latency, and promote the best one.


In [ ]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient()
experiment = mlflow.get_experiment_by_name(
    f"/Users/{username}/data_quality_agent_experiment"
)

# Find the run with the lowest latency that produced enough rules
best_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="metrics.rules_generated >= 5",
    order_by=["metrics.latency_seconds ASC"],
    max_results=1
)

best = best_runs[0]
print(f"Best run: {best.info.run_id}")
print(f"  Latency: {best.data.metrics['latency_seconds']:.2f}s")
print(f"  Rules:   {int(best.data.metrics['rules_generated'])}")


## 3. Promote the Exact Version the Winning Run Evaluated

The alias update is what causes the production serving endpoint to pick
up the promoted version — no redeployment needed.


In [ ]:
model_name = f"{UC_CATALOG}.{UC_SCHEMA}.data_quality_agent"

if best.data.metrics['latency_seconds'] < 10 and best.data.metrics['rules_generated'] >= 5:
    # Promote the exact version this run evaluated, not simply the latest version
    evaluated_version = int(best.data.params['model_version'])

    client.set_registered_model_alias(model_name, "champion", evaluated_version)
    print(f"Promoted version {evaluated_version} to 'champion' based on experimental criteria")
    print(f"  Best run {best.info.run_id} met thresholds:")
    print(f"    Latency: {best.data.metrics['latency_seconds']:.2f}s < 10s")
    print(f"    Rules: {int(best.data.metrics['rules_generated'])} >= 5")
else:
    print("Best run did not meet promotion criteria")
    print(f"  Latency: {best.data.metrics['latency_seconds']:.2f}s")
    print(f"  Rules: {int(best.data.metrics['rules_generated'])}")
